# Summary

## Codebook errors

**Wave 1:**

- `cc_behaviorchange`: not shown (all responses null)

**Wave 2:**

- `cc_behaviorchange`: shown to both new and repeating

**Wave 3:**

- `ew1_jun`: shown to new
- `cvcc4_will`: shown to both new and repeating
- `cvcc7b`: shown conditional on `GroupInfrastructure` (codebook says `GroupGreenInfrastructure`)

## Wave 2 errors

380: 

- Question not shown, response non-null: `ew1`, `ew_attribution` (subset, 284), `cc2`, `cc13`
- Question shown, response is null: `ew1_apr`, `cc13_apr`

45:
- Question shown, response is null: `cc_pol_RE.research`, `cc_pol_tax`, `cc_pol_car`, `cc_pol_subs`, `cc11`, `cv.priority2`

Note, `cv.priority` has no such nulls.

## Wave 3 errors

7: 

- Question not shown, response non-null: `ew1`, `ew_attribution`, `cc5_world`, `cc5_wealthUS`, `cc5_poorUS`, `cc5_comm`, `cc13`, `cvcc7a`, `cvcc7b` (these two partition the 7 PIDs),
- Question shown, response is null: `ew1_jun`

286:

- Question not shown, response non-null: `ew1_jun`, `ew_attribution_jun` (subset, 187)
- Question shown, response is null: `cc5_world`, `cc5_wealthUS`, `cc5_poorUS`, `cc5_comm`, `cc13`, `cvcc7a`, `cvcc7b` (these two partition the 7 PIDs)


## Wave 5 errors

`c_impact_1`: 

- New: 58 null
- Repeating: 49 null

## Empty-string errors

Non-null, empty-string responses

**Wave 3:**

- Question not show, response non-null:
    - `ew1`: all except the 7 nulls listed above.
    - `ew1_jun` all except the 286 nulls listed above.
    - `cc13`: all except the 7 nulls listed above.
- Question shown, response is null:
    - `ew1_jun`: all except the 7 nulls listed above.
    - `cc13`: all except the 286 nulls listed above.

In [ ]:
import polars as pl

from climate_attitudes.settings import Config, RawDataFile

pl.Config.set_tbl_rows(12)
pl.Config.set_tbl_cols(156)

config = Config(_env_file="../.env")

data = (
    pl.read_parquet(RawDataFile.Waves1to5Responses.filepath(config))
    .filter(pl.col("PID").is_not_null())
    .with_columns(
        pl.col("WAVE").cast(int).alias("wave"),
    )
    .rename({"PID": "participant_id"})
    .with_columns(pl.col("wave").min().over("participant_id").alias("wave_joined"))
    .with_columns(
        pl.when(pl.col("wave") == pl.col("wave_joined"))
        .then(pl.lit("new"))
        .otherwise(pl.lit("repeating"))
        .alias("participant_type")
    )
)

# Question not displayed, but response is not null

## `ew1`

**Wave 2**: Only new. 380 repeating responses. Responses not empty.

In [ ]:
ew1_pids = (
    data.filter(wave=2, participant_type="repeating")
    .select("participant_id", "ew1")
    .filter(
        pl.col("ew1").is_not_null(),
    )
    .select(pl.col("participant_id").cast(int).sort())
)

ew1_pids

**Wave 3**: Only new. All 2493 repeating participants have non-null response. Most of these are empty string. 7 are not empty.

Non-empty PIDs:

- 1501076831
- 1502046148
- 1502057962
- 1502984889
- 1503530801
- 1504894342
- 1554133781

In [ ]:
ew1_w3_pids = (
    data.filter(wave=3, participant_type="repeating")
    .select("participant_id", "ew1")
    .filter(
        pl.col("ew1").is_not_null(),
        pl.col("ew1") != "",
    )
    .select(pl.col("participant_id").cast(int).sort())
)

ew1_w3_pids

## `ew1_jun`

**Wave 3**: Only repeating. All 1282 new participants have non-null. 286 of these are not empty.

In [ ]:
ew1_jun_pids = (
    data.filter(wave=3, participant_type="new")
    .select("participant_id", "ew1_jun")
    .filter(
        pl.col("ew1_jun").is_not_null(),
        pl.col("ew1_jun") != "",
    )
    .select(pl.col("participant_id").cast(int).sort())
)
ew1_jun_pids

## `ew_attribution`

**Wave 2**: Only new. 284 repeating responses have non-null. These are a subset of ew1 null PIDs.

In [ ]:
ew_attribution = (
    data.filter(wave=2, participant_type="repeating")
    .select(
        "participant_id",
        "ew_attribution",
        "ew1",
    )
    .filter(
        pl.col("ew_attribution").is_not_null(),
        ~pl.col("ew1").str.split(",").list.contains("0"),
    )
)

ew_attribution

In [ ]:
set(ew_attribution.select(pl.col("participant_id").cast(int)).to_series()).issubset(
    set(ew1_pids.to_series())
)

**Wave 3**: Only new. 4 non-null repeating.

**PIDs**:

- 1501076831
- 1502046148
- 1503530801
- 1504894342

Subset of `cc5_X` ones.

In [ ]:
data.filter(wave=3, participant_type="repeating").select(
    "participant_id",
    "ew_attribution",
    "ew1",
).filter(
    pl.col("ew_attribution").is_not_null(),
    ~pl.col("ew1").str.split(",").list.contains("0"),
).select(pl.col("participant_id").cast(int).sort())

## `ew_attribution_jun`

**Wave 3**: Only repeating. 187 new non-null.

In [ ]:
ew_attribution_jun_pids = (
    data.filter(wave=3, participant_type="new")
    .select(
        "participant_id",
        "ew_attribution_jun",
        "ew1_jun",
    )
    .filter(
        pl.col("ew_attribution_jun").is_not_null(),
        ~pl.col("ew1_jun").str.split(",").list.contains("0"),
    )
    .select(pl.col("participant_id").cast(int).sort())
)
ew_attribution_jun_pids

In [ ]:
set(ew_attribution_jun_pids.to_series()).issubset(set(ew1_jun_pids.to_series()))

## `cc2`

**Wave 2**: Only new. 380 non-null repeating.

In [ ]:
cc2 = (
    data.filter(wave=2, participant_type="repeating")
    .select(
        "participant_id",
        "cc2",
    )
    .filter(
        pl.col("cc2").is_not_null(),
    )
)

## `cc5_world`, `cc5_wealthUS`, `cc5_poorUS`, `cc5_comm`

**Wave 3**: Only new. 7 non-null repeating. 

PIDs:

- 1501076831
- 1502046148
- 1502057962
- 1502984889
- 1503530801
- 1504894342
- 1554133781

In [ ]:
data.filter(wave=3, participant_type="repeating").select(
    "participant_id",
    "cc5_world",
).filter(
    pl.col("cc5_world").is_not_null(),
).select(pl.col("participant_id").cast(int).sort())

## `cc13`

**Wave 2**: Only new. 380 repeating responses. Same participants as in `cc2`.

In [ ]:
cc13_w2 = (
    data.filter(wave=2, participant_type="repeating")
    .select(
        "participant_id",
        "cc13",
    )
    .filter(
        pl.col("cc13").is_not_null(),
    )
)

cc13_w2

In [ ]:
cc2.join(cc13_w2, on="participant_id", how="inner")

**Wave 3**: Only new. All non-null repeating. Only 7 non-empty repeating.

**Non-empty PIDs same as in cc5_X cases.**

In [ ]:
cc13_w3 = (
    data.filter(wave=3, participant_type="repeating")
    .select(
        "participant_id",
        "cc13",
    )
    .filter(
        pl.col("cc13").is_not_null(),
        pl.col("cc13") != "",
    )
)  # .select(pl.col("participant_id").cast(int).sort())

cc13_w3

## `cc_behaviorchange`

**Wave 2**: Neither new nor repeating. All 2377 new respond. All 2817 repeating respond.

In [ ]:
cc_behaviorchange = (
    data.filter(wave=2, participant_type="repeating")
    .select(
        "participant_id",
        "cc_behaviorchange",
    )
    .filter(
        pl.col("cc_behaviorchange").is_not_null(),
        # pl.col("cc13") != "",
    )
)  # .select(pl.col("participant_id").cast(int).sort())

cc_behaviorchange

## `cvcc4_will`

**Wave 3**: Neither new nor repeating. All new respond. All repeating respond.

In [ ]:
cvcc4_will = (
    data.filter(wave=3, participant_type="repeating")
    .select(
        "participant_id",
        "cvcc4_will",
    )
    .filter(
        pl.col("cvcc4_will").is_not_null(),
        # pl.col("cc13") != "",
    )
)  # .select(pl.col("participant_id").cast(int).sort())

cvcc4_will

## `cvcc7a`

**Wave 3**: Only new. 4 repeating non-null. 

**PIDs**:

- 1501076831
- 1502057962
- 1504894342
- 1554133781

Subset of `cc5_X` ones.

In [ ]:
cvcc7a = (
    data.filter(
        wave=3,
        participant_type="repeating",
    )
    .select(
        "participant_id",
        "cvcc7a",
    )
    .filter(
        pl.col("cvcc7a").is_not_null(),
        # pl.col("cc13") != "",
    )
    .select(pl.col("participant_id").cast(int).sort())
)

cvcc7a

## `cvcc7b`

**Wave 3**: Only new. 3 non-null repeating. 

**PIDs**:

- 1502046148
- 1502984889
- 1503530801

Subset of `cc5_X` ones.



In [ ]:
cvcc7b = (
    data.filter(
        wave=3,
        participant_type="repeating",
    )
    .select(
        "participant_id",
        "cvcc7b",
    )
    .filter(
        pl.col("cvcc7b").is_not_null(),
        # pl.col("cc13") != "",
    )
)  # .select(pl.col("participant_id").cast(int).sort())

cvcc7b

# Question displayed but response is null

## `ew1`

**Wave 3**: Only new. Most of repeating are non-null but empty string. (Missing PIDs are the seven from `cc5_X` above).

In [ ]:
ew1 = (
    data.filter(wave=3, participant_type="repeating")
    .select(
        "participant_id",
        "ew1",
    )
    .filter(
        pl.col("ew1").is_null() | (pl.col("ew1") == ""),
        # pl.col("cc13") != "",
    )
    .select(pl.col("participant_id").cast(int).sort())
)

ew1

In [ ]:
set(cc13_w3.to_series()) & set(ew1.to_series())

## `ew1_apr`

**Wave 2**: Only repeating. 380 null repeating. Same 380 participants as in `cc13` (wave 2) and `cc2`.

In [ ]:
ew1_apr = (
    data.filter(wave=2, participant_type="repeating")
    .select(
        "participant_id",
        "ew1_apr",
    )
    .filter(
        pl.col("ew1_apr").is_null()  # | (pl.col("ew1_apr") == ""),
        # pl.col("cc13") != "",
    )
)  # .select(pl.col("participant_id").cast(int).sort())

ew1_apr

In [ ]:
ew1_apr.join(cc13_w2, on="participant_id", how="inner")

## `ew1_jun`

**Wave 3**: Only repeating. 7 null repeating. Same PIDs as in `cc5_X`.

In [ ]:
ew1_jun = (
    data.filter(wave=3, participant_type="repeating")
    .select(
        "participant_id",
        "ew1_jun",
    )
    .filter(
        pl.col("ew1_jun").is_null() | (pl.col("ew1_jun") == ""),
    )
    .select(pl.col("participant_id").cast(int).sort())
)

ew1_jun

## `cc_impact_1`

**Wave 5**: Shown to both new and repeating. 58 null new. 49 null repeating.

In [ ]:
cc_impact_1 = (
    data.filter(wave=5, participant_type="repeating")
    .select(
        "participant_id",
        "cc_impact_1",
    )
    .filter(
        pl.col("cc_impact_1").is_null()  # | (pl.col("ew1_jun") == ""),
    )
    .select(pl.col("participant_id").cast(int).sort())
)

cc_impact_1

## `cc5_world`, `cc5_wealthUS`, `cc5_poorUS`, `cc5_comm`

**Wave 3**: Only new. 286 new are null. Same PIDs as in `ew1_jun` error above (question not displayed but answer not null).

In [ ]:
cc5_world = (
    data.filter(wave=3, participant_type="new")
    .select(
        "participant_id",
        "cc5_world",
    )
    .filter(
        pl.col("cc5_world").is_null()  # | (pl.col("ew1_jun") == ""),
    )
    .select(pl.col("participant_id").cast(int).sort())
)

cc5_world

In [ ]:
len(set(cc5_world.to_series()) & set(ew1_jun_pids.to_series()))

## `cc_pol_RE.research`, `cc_pol_tax`, `cc_pol_car`, `cc_pol_subs`, `cc11`

**Wave 2**: shown to both new and repeating. Repeating has 45 null responses.

In [ ]:
cc_pol_RE__research = (
    data.filter(wave=2, participant_type="repeating")
    .select(
        "participant_id",
        "cc_pol_RE.research",
    )
    .filter(
        pl.col(
            "cc_pol_RE.research"
        ).is_null()  # | (pl.col("cc_pol_RE.research") == ""),
    )
    .select(pl.col("participant_id").cast(int).sort())
)

cc_pol_RE__research

## `cc13`

**Wave 3**: Only new. 286 new responses are null. Same PIDs as in `cc5_X` and `ew1_jun` (from first section).

In [ ]:
cc13 = (
    data.filter(wave=3, participant_type="new")
    .select(
        "participant_id",
        "cc13",
    )
    .filter(pl.col("cc13").is_null() | (pl.col("cc13") == ""))
    .select(pl.col("participant_id").cast(int).sort())
)

cc13

In [ ]:
len(set(cc13.to_series()) & set(ew1_jun_pids.to_series()))

## `cc13_apr`

**Wave 2**: Only repeating. 380 null repeating responses. Same PIDs as in `cc13` (wave 2) and `ew1_apr`.

In [ ]:
cc13_apr = (
    data.filter(wave=2, participant_type="repeating")
    .select(
        "participant_id",
        "cc13_apr",
    )
    .filter(
        pl.col("cc13_apr").is_null()  # | (pl.col("cc13_apr") == "")
    )
    .select(pl.col("participant_id").cast(int).sort())
)

cc13_apr

In [ ]:
cc13_apr.join(
    cc13_w2.with_columns(pl.col("participant_id").cast(int)),
    on="participant_id",
    how="inner",
)

## `cc_behaviorchange`

**Wave 1**: Only new. All responses are null.

In [ ]:
cc_behaviorchange = (
    data.filter(wave=1, participant_type="new")
    .select(
        "participant_id",
        "cc_behaviorchange",
    )
    .filter(
        pl.col("cc_behaviorchange").is_null()  # | (pl.col("cc13_apr") == "")
    )
    .select(pl.col("participant_id").cast(int).sort())
)

cc_behaviorchange

## `cv.priority2`

**Wave 2**: Shown to both new and repeating. 45 null repeating responses. Same PIDs as in `cc_pol_RE.research` and others.

In [ ]:
cv__priority2 = (
    data.filter(wave=2, participant_type="repeating")
    .select(
        "participant_id",
        "cv.priority2",
    )
    .filter(
        pl.col("cv.priority2").is_null()  # | (pl.col("cv.priority2") == "")
    )
    .select(pl.col("participant_id").cast(int).sort())
)

cv__priority2

In [ ]:
len(set(cv__priority2.to_series()) & set(cc_pol_RE__research.to_series()))

# Conditional questions: displayed but response is null.

## `cvcc7a`

**Wave 3**: Only new. Iff `GroupGreenInfrastructure` is True. 146 null new responses. PIDs are subset of those in `cc13` and others.

In [ ]:
cvcc7a = (
    data.filter(
        wave=3,
        participant_type="new",
        GroupGreenInfrastructure=1,
    )
    .select(
        "participant_id",
        "cvcc7a",
    )
    .filter(
        pl.col("cvcc7a").is_null()  # | (pl.col("cv.priority2") == "")
    )
)

cvcc7a

In [ ]:
set(cvcc7a.select(pl.col("participant_id").cast(int)).to_series()).issubset(
    set(cc13.to_series())
)

## `cvcc7b`

**Wave 3**: Only new. Iff `GroupGreenInfrastructure` is True. 626 new null responses.

If condition is changed to `GroupInfrastructure` is True, then 140 new null responses. PIDs subset of those in `cc13` and others. `cvcc7a` and `cvcc7b` null PIDs partition those from `cc13`.

In [ ]:
cvcc7b = (
    data.filter(
        wave=3,
        participant_type="new",
        GroupGreenInfrastructure=1,
    )
    .select(
        "participant_id",
        "cvcc7b",
    )
    .filter(
        pl.col("cvcc7b").is_null()  # | (pl.col("cv.priority2") == "")
    )
)

cvcc7b

In [ ]:
cvcc7b = (
    data.filter(
        wave=3,
        participant_type="new",
        GroupInfrastructure=1,
    )
    .select(
        "participant_id",
        "cvcc7b",
    )
    .filter(
        pl.col("cvcc7b").is_null()  # | (pl.col("cv.priority2") == "")
    )
)

cvcc7b

In [ ]:
set(cvcc7b.select(pl.col("participant_id").cast(int)).to_series()).issubset(
    set(cc13.to_series())
)

In [ ]:
cvcc7a_pids = set(cvcc7a.select(pl.col("participant_id").cast(int)).to_series())
cvcc7b_pids = set(cvcc7b.select(pl.col("participant_id").cast(int)).to_series())
cc13_pids = set(cc13.to_series())
cvcc7a_pids | cvcc7b_pids == cc13_pids

# Null PID respondents from wave 1

In [ ]:
data = (
    pl.read_parquet(RawDataFile.Waves1to5Responses.filepath(config))
    .with_columns(
        pl.col("WAVE").cast(int).alias("wave"),
    )
    .rename({"PID": "participant_id"})
    .with_columns(pl.col("wave").min().over("participant_id").alias("wave_joined"))
    .with_columns(
        pl.when(pl.col("wave") == pl.col("wave_joined"))
        .then(pl.lit("new"))
        .otherwise(pl.lit("repeating"))
        .alias("participant_type")
    )
    .with_columns(
        pl.col("RESPONDENT_TYPE")
        .str.extract(r"W(\d).*", 1)
        .cast(int)
        .alias("extracted_wave_joined")
    )
)

In [ ]:
data.filter(pl.col("wave_joined") != pl.col("extracted_wave_joined"))

In [ ]:
data.filter(
    # pl.col("participant_id").is_null(),
    pl.col("REPEATER") != 0.0,
    # pl.col("participant_type")
    # wave=3,
    pl.col("wave") != 1,
).select(pl.col("participant_id").cast(int))

In [ ]:
data.filter(
    # pl.col("participant_id").is_null(),
    pl.col("REPEATER") != 0.0,
    # pl.col("participant_type")
    pl.col("wave") != 1,
)

In [ ]:
data.filter(
    wave=1,
    participant_id=1067064865,
)